In [1]:
# ==== 3D MRI + mask quick viewer (Training_Set only) =========================
# Paths
from pathlib import Path
IMAGES_DIR = Path("//home/rbielski/Atlas_2/Training/Images")
MASKS_DIR  = Path("/home/rbielski/Atlas_2/Training/Masks")

import os, math, logging
import numpy as np
import nibabel as nib
import matplotlib.pyplot as plt
from functools import lru_cache
import ipywidgets as W
from IPython.display import display, clear_output

# Quiet down nibabel "qfac" chatter
logging.getLogger("nibabel").setLevel(logging.ERROR)

# ---------------- pairing helpers (fit your filenames) -----------------------
def _img_id(name: str) -> str:
    """ID from image file name, e.g.
    sub-xxx_ses-1_space-..._T1w.nii.gz  ->  sub-xxx_ses-1_space-...
    """
    base = name
    if base.endswith(".nii.gz"): base = base[:-7]
    # strip common image suffixes at the end
    for suf in ["_T1w", "_t1", "_T2w", "_FLAIR", "_image", "_brain"]:
        if base.endswith(suf):
            base = base[: -len(suf)]
            break
    return base

def _mask_id(name: str) -> str:
    """ID from mask file name, e.g.
    sub-xxx_ses-1_space-..._label-L_desc-T1lesion_mask.nii.gz
      -> sub-xxx_ses-1_space-...
    """
    base = name
    if base.endswith(".nii.gz"): base = base[:-7]
    for tok in ["_label", "_lesion", "_mask", "_seg"]:
        i = base.find(tok)
        if i != -1:
            base = base[:i]
            break
    return base

def build_pairs(images_dir: Path, masks_dir: Path):
    imgs = { _img_id(p.name): p for p in sorted(images_dir.glob("*.nii.gz")) }
    msks = { _mask_id(p.name): p for p in sorted(masks_dir.glob("*.nii.gz")) }
    common = sorted(set(imgs).intersection(msks))
    pairs = [(imgs[k], msks[k]) for k in common]
    return pairs, len(imgs), len(msks), len(common)

pairs, n_img, n_msk, n_pair = build_pairs(IMAGES_DIR, MASKS_DIR)

print(f"Found images: {n_img} | masks: {n_msk} | paired: {n_pair}")
if n_pair == 0:
    raise RuntimeError(
        "No pairs found in Training_Set. Check that files exist in:\n"
        f"- {IMAGES_DIR}\n- {MASKS_DIR}\n"
        "and that image IDs (before _T1w) match mask IDs (before _label/_mask)."
    )

# ---------------- caching loaders & utilities --------------------------------
@lru_cache(maxsize=64)
def _load_nii(path: str):
    img = nib.load(path)
    data = img.get_fdata()
    return data  # float64/float32 depending on file

def _norm01(x, invert=False):
    x = np.asarray(x, dtype=np.float32)
    # robust [p2, p98] window
    p2, p98 = np.percentile(x[np.isfinite(x)], [2, 98])
    if p98 <= p2:
        p2, p98 = x.min(), x.max()
    x = np.clip((x - p2) / max(1e-6, (p98 - p2)), 0, 1)
    if invert: x = 1.0 - x
    return x

def _slice2d(vol, axis, idx):
    if axis == 2:   # axial (z)
        return vol[:, :, idx]
    elif axis == 1: # coronal (y)
        return vol[:, idx, :]
    else:           # sagittal (x)
        return vol[idx, :, :]

def _edges2d(m):
    # simple 2D edge mask via dilation difference (fast)
    from scipy.ndimage import binary_dilation
    m = m.astype(bool)
    return binary_dilation(m) & (~m)

# ---------------- widgets -----------------------------------------------------
split_label = W.HTML(f"<b>Training_Set only</b> — {n_pair} pairs found")

axis_rb = W.RadioButtons(
    options=[("Axial (z)", 2), ("Coronal (y)", 1), ("Sagittal (x)", 0)],
    value=2, description="Axis:"
)

# Dropdown options: nice label, real (img,mask) tuple as value
def _option_label(img_path, msk_path):
    # show the shared ID (before suffix)
    return os.path.basename(msk_path).split("_label")[0]

pair_dd = W.Dropdown(
    options=[(_option_label(i, m), (str(i), str(m))) for (i, m) in pairs],
    description="Pair:",
    layout=W.Layout(width="100%")
)

slice_sl = W.IntSlider(description="Slice:", min=0, max=1, value=0, continuous_update=False)

mask_alpha = W.FloatSlider(description="Mask α:", min=0.0, max=1.0, step=0.05, value=0.55)
edges_only = W.Checkbox(description="Edges only (faster/clearer)", value=True)
invert_img = W.Checkbox(description="Invert image", value=False)
status = W.HTML("Viewer ready.")

controls = W.VBox([
    split_label,
    W.HBox([axis_rb, slice_sl]),
    pair_dd,
    W.HBox([mask_alpha, edges_only, invert_img]),
    status,
])

out = W.Output()

# ---------------- reactive update --------------------------------------------
def _update_slider_range(*_):
    img_path, msk_path = pair_dd.value
    vol = _load_nii(img_path)
    ax  = axis_rb.value
    max_idx = int(vol.shape[ax] - 1)
    slice_sl.max = max(0, max_idx)
    # keep current value in range
    slice_sl.value = min(slice_sl.value, slice_sl.max)

def _redraw(*_):
    with out:
        clear_output(wait=True)
        try:
            img_path, msk_path = pair_dd.value
            vol = _load_nii(img_path)
            msk = _load_nii(msk_path)
            ax  = axis_rb.value
            idx = int(slice_sl.value)

            if vol.shape[:3] != msk.shape[:3]:
                status.value = (f"<span style='color:#e55'>Shape mismatch: "
                                f"{vol.shape[:3]} vs {msk.shape[:3]}</span>")
            else:
                status.value = " "

            img2d = _slice2d(vol, ax, idx)
            m2d   = _slice2d(msk, ax, idx) > 0

            img2d = _norm01(img2d, invert=invert_img.value)

            plt.figure(figsize=(6,6))
            plt.imshow(img2d.T, cmap="gray", origin="lower")
            if edges_only.value:
                e = _edges2d(m2d)
                plt.contour(e.T, levels=[0.5], linewidths=0.7, colors="r")
            else:
                plt.imshow(np.ma.masked_where(~m2d.T, m2d.T), cmap="jet", alpha=float(mask_alpha.value), origin="lower")
            plt.axis("off")
            plt.show()
        except Exception as e:
            status.value = f"<span style='color:#e55'>Error: {e}</span>"

# wire up events
pair_dd.observe(_update_slider_range, names="value")
pair_dd.observe(_redraw, names="value")
axis_rb.observe(_update_slider_range, names="value")
axis_rb.observe(_redraw, names="value")
slice_sl.observe(_redraw, names="value")
invert_img.observe(_redraw, names="value")
edges_only.observe(_redraw, names="value")
mask_alpha.observe(_redraw, names="value")

# initial slider range & draw
_update_slider_range()
_redraw()

display(controls, out)
# ============================================================================== 


Found images: 655 | masks: 655 | paired: 655


Output()

In [3]:
from templateflow.api import get as tf_get
from pathlib import Path
import nibabel as nib

# Option A: full-head 1mm T1w (usually a single file)
mni_files = tf_get("MNI152NLin2009cAsym", resolution=1, suffix="T1w", extension="nii.gz")
MNI_PATH = Path(mni_files[0])  # <-- pick the first (or add logic below)
print("MNI:", MNI_PATH)

img = nib.load(str(MNI_PATH))
print("shape:", img.shape, "zooms:", img.header.get_zooms())


# Quick sanity check: ANTs + Template present + first pair exists
import shutil, os
from pathlib import Path

ANTS_REG   = shutil.which("antsRegistration") or "/home/rbielski/miniconda3/envs/stroke_env/bin/antsRegistration"
ANTS_APPLY = shutil.which("antsApplyTransforms") or "/home/rbielski/miniconda3/envs/stroke_env/bin/antsApplyTransforms"

mask = Path("/home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/sub-M2012_ses-1158_acq-spc3_run-4_T2w_desc-lesion_mask.nii.gz")
t1   = Path("/home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/sub-M2012_ses-1158_acq-tfl3p2_run-3_T1w.nii.gz")

print("antsRegistration:", ANTS_REG, "exists:", Path(ANTS_REG).exists())
print("antsApplyTransforms:", ANTS_APPLY, "exists:", Path(ANTS_APPLY).exists())
print("Mask exists:", mask.exists())
print("T1w  exists:", t1.exists())

# TemplateFlow cache path you previously used:
mni = Path("/home/rbielski/.cache/templateflow/tpl-MNI152NLin2009cAsym/tpl-MNI152NLin2009cAsym_res-01_desc-brain_T1w.nii.gz")
print("MNI template:", mni, "exists:", mni.exists())

# Optional: print ANTs version
import subprocess
try:
    out = subprocess.run([ANTS_REG, "--version"], capture_output=True, text=True)
    print("\nantsRegistration --version:\n", out.stdout or out.stderr)
except Exception as e:
    print("Version check error:", e)


MNI: /home/rbielski/.cache/templateflow/tpl-MNI152NLin2009cAsym/tpl-MNI152NLin2009cAsym_res-01_desc-brain_T1w.nii.gz
shape: (193, 229, 193) zooms: (1.0, 1.0, 1.0)
antsRegistration: /home/rbielski/miniconda3/envs/stroke_env/bin/antsRegistration exists: True
antsApplyTransforms: /home/rbielski/miniconda3/envs/stroke_env/bin/antsApplyTransforms exists: True
Mask exists: True
T1w  exists: True
MNI template: /home/rbielski/.cache/templateflow/tpl-MNI152NLin2009cAsym/tpl-MNI152NLin2009cAsym_res-01_desc-brain_T1w.nii.gz exists: True

antsRegistration --version:
 ANTs Version: 2.6.0.dev1-gb775a15
Compiled: Apr 16 2025 14:14:03




In [4]:
# === Batch: (1) Mask->T1 (NN, bin)  (2) T1->MNI SyN  (3) Apply to T1+mask ===
import os, sys, re, csv, site, shutil, subprocess
from pathlib import Path
from datetime import datetime
import nibabel as nib
import numpy as np
from nibabel.processing import resample_from_to

# ---------- PATHS ----------
TRAIN_ROOT = Path("/home/rbielski/Atlas_2/Training")
IMAGES_DIR = TRAIN_ROOT / "Images"
MASKS_DIR = TRAIN_ROOT / "Masks"

if not IMAGES_DIR.exists() or not MASKS_DIR.exists():
    raise FileNotFoundError("Expected Images/ and Masks/ under /home/rbielski/Atlas_2/Training")

OUT_ROOT = Path("/home/rbielski/Atlas_2/Registered")
OUT_TMP = OUT_ROOT / "._ants_work2"               # transforms / intermediates
OUT_NAT = OUT_ROOT / "native_resampled_masks"     # masks resampled to native T1 grid
OUT_MNI = OUT_ROOT / "mni_1mm_ants_fixed"         # final MNI outputs (T1 + mask)
for d in (OUT_ROOT, OUT_TMP, OUT_NAT, OUT_MNI):
    d.mkdir(exist_ok=True, parents=True)

# ---------- SPEED SETTINGS ----------
# Use 2mm template for registration (big speedup), then apply transforms at 1mm for outputs.
USE_2MM_FOR_REG = True
# Threading for ANTs/ITK (pick what your box can handle)
os.environ.setdefault("ITK_GLOBAL_DEFAULT_NUMBER_OF_THREADS", "8")

# ---------- TEMPLATE ----------
try:
    from templateflow.api import get as tf_get
except Exception:
    subprocess.run([sys.executable, "-m", "pip", "install", "--user", "templateflow"], check=True)
    sys.path.append(site.getusersitepackages())
    from templateflow.api import get as tf_get

tpl_id = "MNI152NLin2009cAsym"
mni_1mm = tf_get(tpl_id, resolution=1, suffix="T1w", desc="brain", extension="nii.gz")
MNI_1MM = Path(mni_1mm[0] if isinstance(mni_1mm, (list, tuple)) else mni_1mm)
assert MNI_1MM.exists(), f"Missing template: {MNI_1MM}"

if USE_2MM_FOR_REG:
    mni_2mm = tf_get(tpl_id, resolution=2, suffix="T1w", desc="brain", extension="nii.gz")
    MNI_REG = Path(mni_2mm[0] if isinstance(mni_2mm, (list, tuple)) else mni_2mm)
else:
    MNI_REG = MNI_1MM
assert Path(MNI_REG).exists(), f"Missing registration template: {MNI_REG}"

# ---------- ANTs ----------
ANTS_REG   = shutil.which("antsRegistration")    or "/home/rbielski/miniconda3/envs/stroke_env/bin/antsRegistration"
ANTS_APPLY = shutil.which("antsApplyTransforms") or "/home/rbielski/miniconda3/envs/stroke_env/bin/antsApplyTransforms"
assert Path(ANTS_REG).exists(),   f"antsRegistration not found: {ANTS_REG}"
assert Path(ANTS_APPLY).exists(), f"antsApplyTransforms not found: {ANTS_APPLY}"

def run(cmd, check=True):
    # Add --float 1 for speed/memory where applicable
    if cmd[0].endswith("antsRegistration") and "--float" not in cmd:
        cmd += ["--float", "1"]
    if cmd[0].endswith("antsApplyTransforms") and "--float" not in cmd:
        cmd += ["--float", "1"]
    print(">>", " ".join(cmd))
    res = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    if check and res.returncode != 0:
        print(res.stdout)
        raise RuntimeError("Command failed")
    return res.stdout

# ---------- HELPERS ----------
def _tag(s, tag):
    m = re.search(fr"({tag}-[^_]+)", s)
    return m.group(1) if m else None

def key_from_path(p: Path) -> str:
    sub = _tag(p.name, "sub")
    ses = _tag(p.name, "ses")
    return "_".join([x for x in (sub, ses) if x])

def _t1_pref(name: str) -> int:
    n = name.lower()
    if "tfl" in n: return 0
    if "mprage" in n: return 1
    if "mp2rage" in n: return 2
    return 3

def choose_t1_for(key: str) -> Path | None:
    patterns = (f"{key}_*T1w.nii.gz", f"{key}*T1w.nii.gz")
    for pattern in patterns:
        cands = sorted(IMAGES_DIR.glob(pattern))
        if cands:
            cands.sort(key=lambda p: (_t1_pref(p.name), p.name))
            return cands[0]
    return None

def resample_mask_to_t1(mask_path: Path, t1_path: Path, out_path: Path):
    mi = nib.load(str(mask_path))
    ti = nib.load(str(t1_path))
    if mi.shape[:3] != ti.shape[:3] or not np.allclose(mi.affine, ti.affine, atol=1e-4):
        rs = resample_from_to(mi, (ti.shape, ti.affine), order=0)   # NN resample
        data = (rs.get_fdata() > 0.5).astype(np.uint8)
    else:
        data = (mi.get_fdata() > 0.5).astype(np.uint8)
    nib.save(nib.Nifti1Image(data, ti.affine, ti.header), str(out_path))

def ants_syn_t1_to_mni(t1_path: Path, prefix: Path):
    # One SyN per key; register to MNI_REG (2mm if enabled)
    run([
        ANTS_REG, "-d","3",
        "-r", f"[{MNI_REG},{t1_path},1]",
        "-m", f"Mattes[{MNI_REG},{t1_path},1,32,Regular,0.25]",
        "-t","Rigid[0.1]","-c","1000x500x250","-s","3x2x1vox","-f","4x2x1",
        "-m", f"Mattes[{MNI_REG},{t1_path},1,32,Regular,0.25]",
        "-t","Affine[0.1]","-c","1000x500x250","-s","3x2x1vox","-f","4x2x1",
        "-m", f"CC[{MNI_REG},{t1_path},1,4]",
        "-t","SyN[0.1,3,0]","-c","60x40x20","-s","2x1x0vox","-f","4x2x1",  # slightly faster schedule
        "-o", f"[{prefix},{prefix}warped.nii.gz,{prefix}invwarped.nii.gz]"
    ])

def apply_to(img_in: Path, ref: Path, xfm_prefix: Path, out_path: Path, nn=False):
    args = [ANTS_APPLY, "-d","3", "-i", str(img_in), "-r", str(ref), "-o", str(out_path)]
    if nn: args += ["-n","NearestNeighbor"]
    args += ["-t", str(xfm_prefix) + "1Warp.nii.gz", "-t", str(xfm_prefix) + "0GenericAffine.mat"]
    run(args)

# ---------- DISCOVER (unique masks) ----------
raw_masks = sorted(MASKS_DIR.glob("*_mask.nii.gz"))
assert raw_masks, f"No lesion masks found under {MASKS_DIR}"
# De-dup by full filename; keeps stability if the folder has dupes/symlinks
seen = {}
for p in raw_masks:
    seen[p.name] = p
native_masks = list(seen.values())
print(f"[info] masks found in {MASKS_DIR}: {len(raw_masks)} | unique: {len(native_masks)}")

# ---------- MAIN LOOP (idempotent) ----------
qc_rows = []
ts = datetime.now().isoformat(timespec="seconds")

for m in native_masks:
    key = key_from_path(m)
    t1 = choose_t1_for(key)
    if not t1:
        print(f"[skip] No T1 found in {IMAGES_DIR} for {m.name}")
        continue

    out_mask_t1 = OUT_NAT / f"{key}_lesion_mask_T1w_native.nii.gz"
    xfm_prefix  = OUT_TMP / f"{key}_t1_to_mni_"
    warp_file   = OUT_TMP / f"{key}_t1_to_mni_1Warp.nii.gz"
    aff_file    = OUT_TMP / f"{key}_t1_to_mni_0GenericAffine.mat"
    out_t1_mni  = OUT_MNI / f"{key}_T1w_MNI.nii.gz"
    out_msk_mni = OUT_MNI / f"{key}_lesion_mask_MNI.nii.gz"

    # (1) Mask→T1 grid (NN + bin)
    if not out_mask_t1.exists():
        print(f"\n=== {key} :: resample mask→T1 ===")
        print("T1 :", t1.name)
        print("MSK:", m.name)
        resample_mask_to_t1(m, t1, out_mask_t1)

    # (2) T1→MNI SyN (once per key)
    if not (warp_file.exists() and aff_file.exists()):
        print(f"=== {key} :: antsRegistration (T1→MNI) ===")
        ants_syn_t1_to_mni(t1, xfm_prefix)

    # (3) Apply transforms (T1: linear interp default; Mask: NN)
    if not out_t1_mni.exists():
        apply_to(t1, MNI_1MM, xfm_prefix, out_t1_mni, nn=False)
    if not out_msk_mni.exists():
        apply_to(out_mask_t1, MNI_1MM, xfm_prefix, out_msk_mni, nn=True)
        # hard re-binarize (paranoia)
        mi = nib.load(str(out_msk_mni))
        data = (mi.get_fdata() > 0.5).astype(np.uint8)
        nib.save(nib.Nifti1Image(data, mi.affine, mi.header), str(out_msk_mni))

    # QC row
    ti = nib.load(str(out_t1_mni)); mi = nib.load(str(out_msk_mni))
    qc_rows.append(dict(
        key=key,
        t1_src=str(t1), mask_src=str(m),
        mask_native=str(out_mask_t1),
        t1_mni=str(out_t1_mni), mask_mni=str(out_msk_mni),
        t1_shape=str(ti.shape), t1_zooms=str(tuple(round(z,3) for z in ti.header.get_zooms()[:3])),
        mask_shape=str(mi.shape), mask_zooms=str(tuple(round(z,3) for z in mi.header.get_zooms()[:3])),
        mask_nonzero=int(np.count_nonzero(mi.get_fdata() > 0)),
        timestamp=ts
    ))

# ---------- WRITE QC ----------
qc_csv = OUT_MNI / "qc_summary.csv"
if qc_rows:
    with open(qc_csv, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=list(qc_rows[0].keys()))
        w.writeheader(); w.writerows(qc_rows)
    print("\nQC written:", qc_csv)
else:
    print("\nNo QC rows written.")


[info] masks found in /home/rbielski/Atlas_2/Training/Masks: 655 | unique: 655

=== sub-r001s001_ses-1 :: resample mask→T1 ===
T1 : sub-r001s001_ses-1_space-MNI152NLin2009aSym_T1w.nii.gz
MSK: sub-r001s001_ses-1_space-MNI152NLin2009aSym_label-L_desc-T1lesion_mask.nii.gz
=== sub-r001s001_ses-1 :: antsRegistration (T1→MNI) ===
>> /home/rbielski/miniconda3/envs/stroke_env/bin/antsRegistration -d 3 -r [/home/rbielski/.cache/templateflow/tpl-MNI152NLin2009cAsym/tpl-MNI152NLin2009cAsym_res-02_desc-brain_T1w.nii.gz,/home/rbielski/Atlas_2/Training/Images/sub-r001s001_ses-1_space-MNI152NLin2009aSym_T1w.nii.gz,1] -m Mattes[/home/rbielski/.cache/templateflow/tpl-MNI152NLin2009cAsym/tpl-MNI152NLin2009cAsym_res-02_desc-brain_T1w.nii.gz,/home/rbielski/Atlas_2/Training/Images/sub-r001s001_ses-1_space-MNI152NLin2009aSym_T1w.nii.gz,1,32,Regular,0.25] -t Rigid[0.1] -c 1000x500x250 -s 3x2x1vox -f 4x2x1 -m Mattes[/home/rbielski/.cache/templateflow/tpl-MNI152NLin2009cAsym/tpl-MNI152NLin2009cAsym_res-02_desc

In [ ]:
# ==== MNI audit + T1w/MNI viewer (auto-detect output dirs) ====
from pathlib import Path
import re, numpy as np, nibabel as nib
import matplotlib.pyplot as plt
import ipywidgets as W
from IPython.display import display, clear_output
from functools import lru_cache
from scipy.ndimage import binary_dilation



# BASE (the directory that directly contains mni_1mm_ants/ and/or mni_1mm_ants_fixed/)
ROOT = Path("/home/rbielski/Atlas_2/Registered")
TRAIN_IMAGES = Path("/home/rbielski/Atlas_2/Training/Images")
CANDIDATE_MNI_DIRS = [ROOT / "mni_1mm_ants_fixed", ROOT / "mni_1mm_ants"]
OUT_MNI = next((p for p in CANDIDATE_MNI_DIRS if p.exists()), None)
print("ROOT:", ROOT)
print("Found OUT_MNI:", OUT_MNI)


OUT_NATMSK = ROOT / "native_resampled_masks"  # optional
HAVE_NATIVE = OUT_NATMSK.exists()

TPL = Path("/home/rbielski/.cache/templateflow/tpl-MNI152NLin2009cAsym/tpl-MNI152NLin2009cAsym_res-01_desc-brain_T1w.nii.gz")
tpl_img = nib.load(str(TPL))
tpl_shape = tpl_img.shape[:3]
tpl_zooms = tuple(float(z) for z in tpl_img.header.get_zooms()[:3])
tpl_aff   = tpl_img.affine

def _key_from_name(p: Path) -> str:
    m_sub = re.search(r"(sub-[^_]+)", p.name)
    m_ses = re.search(r"(ses-[^_]+)", p.name)
    parts = [m_sub.group(1) if m_sub else None, m_ses.group(1) if m_ses else None]
    return "_".join([x for x in parts if x])

@lru_cache(maxsize=256)
def _load_img(path: str) -> nib.Nifti1Image: return nib.load(path)

@lru_cache(maxsize=256)
def _load_vol(path: str) -> np.ndarray:
    arr = nib.load(path).get_fdata()
    if arr.ndim == 4 and arr.shape[-1] == 1: arr = arr[..., 0]
    return arr.astype(np.float32)

def _normalize(img: np.ndarray) -> np.ndarray:
    nz = img[img > 0]
    if nz.size == 0: return np.zeros_like(img, dtype=np.float32)
    p1, p99 = np.percentile(nz, [1, 99]); img = np.clip(img, p1, p99)
    m, s = nz.mean(), nz.std(); img = (img - m) / (s + 1e-8)
    mn, mx = img.min(), img.max()
    return (img - mn) / (mx - mn + 1e-8)

def _edges2d(m2d): 
    from scipy.ndimage import binary_dilation
    m = m2d.astype(bool); return binary_dilation(m) & (~m)

def _zooms3(img: nib.Nifti1Image):
    z = img.header.get_zooms()[:3]
    return tuple(float(v) for v in z)

def _affine_equal(a: np.ndarray, b: np.ndarray, tol=1e-4): 
    return np.allclose(a, b, atol=tol)

# ---- Collect MNI pairs ----
t1_mni = sorted(OUT_MNI.glob("*_T1w_MNI.nii.gz"))
mask_mni_by_base = {p.name.replace("_lesion_mask_MNI.nii.gz",""): p
                    for p in OUT_MNI.glob("*_lesion_mask_MNI.nii.gz")}
PAIRS = {}
for t1p in t1_mni:
    key  = _key_from_name(t1p)
    base = t1p.name.replace("_T1w_MNI.nii.gz", "")
    mskp = mask_mni_by_base.get(base)
    if mskp: PAIRS[key] = {"t1_mni": t1p, "mask_mni": mskp}

assert PAIRS, f"No *_T1w_MNI / *_lesion_mask_MNI pairs found in {OUT_MNI}"

# ---- MNI audit ----
bad = []
for k, v in PAIRS.items():
    ti = _load_img(str(v["t1_mni"])); mi = _load_img(str(v["mask_mni"]))
    ok_shape = (ti.shape[:3] == tpl_shape) and (mi.shape[:3] == tpl_shape)
    ok_zooms = (_zooms3(ti) == tpl_zooms) and (_zooms3(mi) == tpl_zooms)
    ok_aff_t1  = _affine_equal(ti.affine, tpl_aff)
    ok_aff_msk = _affine_equal(mi.affine, tpl_aff)
    if not (ok_shape and ok_zooms and ok_aff_t1 and ok_aff_msk):
        bad.append(dict(
            key=k, t1=v["t1_mni"].name, msk=v["mask_mni"].name,
            t1_shape=ti.shape[:3], msk_shape=mi.shape[:3],
            t1_zooms=_zooms3(ti), msk_zooms=_zooms3(mi),
            t1_aff_ok=ok_aff_t1, msk_aff_ok=ok_aff_msk
        ))

print(f"[MNI audit] dir={OUT_MNI.name} | total pairs: {len(PAIRS)} | OK: {len(PAIRS)-len(bad)} | FAIL: {len(bad)}")
if bad:
    for row in bad[:10]: print(row)

# ---- Optional native view wiring (if you created resampled masks) ----
if HAVE_NATIVE:
    for k in list(PAIRS.keys()):
        t1_native = next(iter(TRAIN_IMAGES.glob(f"{k}*T1w.nii.gz")), None)
        m_native  = OUT_NATMSK / f"{k}_lesion_mask_T1w_native.nii.gz"
        if t1_native and m_native.exists():
            PAIRS[k]["t1_native"] = t1_native
            PAIRS[k]["mask_t1"]   = m_native

# ---- Viewer ----
keys_sorted = sorted(PAIRS.keys())
pair_dd   = W.Dropdown(options=keys_sorted, description="Case:", layout=W.Layout(width="100%"))
slice_sl  = W.IntSlider(description="Axial slice:", min=0, max=1, value=0, continuous_update=False, layout=W.Layout(width="60%"))
alpha_sl  = W.FloatSlider(description="Mask α:", min=0.0, max=1.0, step=0.05, value=0.55, layout=W.Layout(width="35%"))
edges_cb  = W.Checkbox(description="Edges only", value=True)
invert_cb = W.Checkbox(description="Invert image", value=False)

def _bg_options_for(key: str):
    opts = [("T1w_MNI (final)", "MNI")]
    if "t1_native" in PAIRS[key] and "mask_t1" in PAIRS[key]:
        opts.append(("T1w_native (with native mask)", "NATIVE"))
    return opts

bg_radio = W.RadioButtons(options=_bg_options_for(keys_sorted[0]), value="MNI",
                          description="Background:", layout=W.Layout(width="40%"))

status = W.HTML(f"<b>Viewer</b> — cases: {len(keys_sorted)}")
controls = W.VBox([status, pair_dd, W.HBox([slice_sl, alpha_sl]), W.HBox([edges_cb, invert_cb, bg_radio])])
out = W.Output()

def _update_bg_options(*_):
    key = pair_dd.value
    bg_radio.options = _bg_options_for(key)
    if bg_radio.value not in [v for _, v in bg_radio.options]:
        bg_radio.value = bg_radio.options[0][1]

def _update_slice_range(*_):
    key = pair_dd.value
    if bg_radio.value == "MNI":
        vol = _load_vol(str(PAIRS[key]["t1_mni"]))
    else:
        vol = _load_vol(str(PAIRS[key]["t1_native"]))
    slice_sl.max = max(0, int(vol.shape[2] - 1))
    slice_sl.value = min(slice_sl.value, slice_sl.max)

def _draw(*_):
    with out:
        clear_output(wait=True)
        try:
            key = pair_dd.value
            if bg_radio.value == "MNI":
                img_p  = PAIRS[key]["t1_mni"];  mask_p = PAIRS[key]["mask_mni"];  bg_name = "T1w_MNI"
                ti = _load_img(str(img_p)); mi = _load_img(str(mask_p))
                aff_ok = _affine_equal(ti.affine, tpl_aff) and _affine_equal(mi.affine, tpl_aff)
                shp_ok = (ti.shape[:3] == tpl_shape) and (mi.shape[:3] == tpl_shape)
                z_ok   = (_zooms3(ti) == tpl_zooms) and (_zooms3(mi) == tpl_zooms)
                audit = f" | MNI check: shape {'✅' if shp_ok else '⚠️'}, zooms {'✅' if z_ok else '⚠️'}, affine {'✅' if aff_ok else '⚠️'}"
            else:
                img_p  = PAIRS[key]["t1_native"]; mask_p = PAIRS[key]["mask_t1"];  bg_name = "T1w_native"; audit = ""

            img_hdr = _load_img(str(img_p)); img = _load_vol(str(img_p))
            msk_hdr = _load_img(str(mask_p)); msk = (_load_vol(str(mask_p)) > 0.5)

            z_img = _zooms3(img_hdr); z_msk = _zooms3(msk_hdr)
            aff_eq = _affine_equal(img_hdr.affine, msk_hdr.affine)
            shp_eq = img_hdr.shape[:3] == msk_hdr.shape[:3]
            status.value = (f"<b>Viewer</b> — cases: {len(keys_sorted)} | {bg_name}: "
                            f"shape {img_hdr.shape[:3]} vs mask {msk_hdr.shape[:3]} "
                            f"| zooms {tuple(round(v,3) for v in z_img)} / {tuple(round(v,3) for v in z_msk)} "
                            f"| affines match: {'✅' if aff_eq else '⚠️'} | shapes match: {'✅' if shp_eq else '⚠️'}{audit}")

            img_view = _normalize(img.copy())
            if invert_cb.value: img_view = 1.0 - img_view
            idx = int(slice_sl.value)
            img2d = img_view[:, :, idx]; m2d = msk[:, :, idx]

            plt.figure(figsize=(5.6, 5.6))
            plt.imshow(img2d.T, cmap="gray", origin="lower")
            if edges_cb.value:
                plt.contour(_edges2d(m2d).T, levels=[0.5], linewidths=0.8, colors="r")
            else:
                plt.imshow(np.ma.masked_where(~m2d.T, m2d.T), cmap="jet", alpha=float(alpha_sl.value), origin="lower")
            plt.axis("off"); plt.tight_layout(); plt.show(); plt.close()

            print(f"Image: {img_p.name}\nMask : {mask_p.name}\nDir: {OUT_MNI}")

        except Exception as exc:
            print("Draw error:", exc)

def _refresh(*_):
    _update_bg_options(); _update_slice_range(); _draw()

pair_dd.observe(_refresh, names="value")
slice_sl.observe(_draw, names="value")
alpha_sl.observe(_draw, names="value")
edges_cb.observe(_draw, names="value")
invert_cb.observe(_draw, names="value")
bg_radio.observe(_refresh, names="value")

_refresh()
display(controls, out)


ROOT: /home/rbielski/Atlas_2/Registered
Found OUT_MNI: /home/rbielski/Atlas_2/Registered/mni_1mm_ants_fixed
[MNI audit] dir=mni_1mm_ants_fixed | total pairs: 655 | OK: 655 | FAIL: 0


Output()

In [6]:
from pathlib import Path
import pandas as pd, numpy as np

OUT = Path("/home/rbielski/Atlas_2/Registered/mni_1mm_ants_fixed")
qc = pd.read_csv(OUT / "qc_summary.csv")

# mask volume (mm^3) from header zooms * voxel count
def parse_tuple(s):
    # strings like "(193, 229, 193)" or "(1.0, 1.0, 1.0)"
    s = s.strip().strip("()")
    return tuple(float(x) for x in s.split(","))
vol_mm3 = []
for z_str, nvox in zip(qc["mask_zooms"], qc["mask_nonzero"]):
    z = parse_tuple(z_str)
    vol_mm3.append(nvox * (z[0]*z[1]*z[2]))
qc["mask_vol_mm3"] = vol_mm3

print("Pairs:", len(qc))
print("Mask voxel counts — min/median/max:", int(qc["mask_nonzero"].min()),
      int(qc["mask_nonzero"].median()), int(qc["mask_nonzero"].max()))
print("Mask volume (ml) — min/median/max:",
      round(qc["mask_vol_mm3"].min()/1000,2),
      round(qc["mask_vol_mm3"].median()/1000,2),
      round(qc["mask_vol_mm3"].max()/1000,2))

# flag anything suspiciously tiny/huge
sus = qc[(qc["mask_vol_mm3"] < 1_000) | (qc["mask_vol_mm3"] > 200_000)]  # <1 ml or >200 ml
print("\nSuspicious volumes:", len(sus))
print(sus[["key","mask_nonzero","mask_vol_mm3"]].head(10).to_string(index=False))


Pairs: 655
Mask voxel counts — min/median/max: 0 3314 163423
Mask volume (ml) — min/median/max: 0.0 3.31 163.42

Suspicious volumes: 193
               key  mask_nonzero  mask_vol_mm3
sub-r001s018_ses-1           186         186.0
sub-r001s022_ses-1            70          70.0
sub-r001s023_ses-1            21          21.0
sub-r001s024_ses-1           556         556.0
sub-r001s027_ses-1           922         922.0
sub-r001s028_ses-1           106         106.0
sub-r001s038_ses-1           146         146.0
sub-r003s006_ses-1           908         908.0
sub-r003s010_ses-1           227         227.0
sub-r005s026_ses-1            46          46.0


#2. How do other people deal with tiny lesions?

Short version:
Most stroke lesion segmentation work on ATLAS v1/v2 does not simply throw tiny lesions away, but everyone agrees they are hard and mess with Dice. People handle them in a few different ways:

2.1. Dataset side: minimum lesion size / metadata

The original ATLAS v1 paper notes a minimum lesion size of 10 mm³ for inclusion in the dataset.
Nature

At 1×1×1 mm, that’s ~10 voxels — already very small.

The ATLAS v2 tracing protocol emphasizes including lesions across a broad size range (small, medium, large) and records lesion size metadata.
Frontiers

→ They don’t discard small lesions post-hoc; instead they keep them and track their sizes.

So the dataset curation may filter out truly microscopic blobs (< a few voxels), but not the “normal sized” small lesions you’re seeing.

2.2. Training strategies for small lesions

Recent work treats small-lesion segmentation as a core problem rather than something to exclude:

Multi-Size Labeling (MSL) and Distance-Based Labeling (DBL)
Shang et al. 2024 (“Segmenting Small Stroke Lesions with Novel Labeling Strategies”) explicitly target small stroke lesions on ATLAS v2.0. They:

split lesion voxels into volume-based classes (tiny / small / medium / large) and

or label voxels by distance to lesion boundary.
This improves Dice and F1 especially on small-lesion subsets.
arXiv
+1

Loss reweighting for lesion size imbalance
Rachmadi et al. 2024 use universal loss reweighting to balance lesion-size inequality in 3D medical image segmentation on ATLAS v2.0. Small lesions get relatively higher weight in the loss so they aren’t overwhelmed by large lesions.
Proceedings of Machine Learning Research

Lesion-aware patch sampling / augmentation
For MS and other lesion tasks, people oversample patches that contain lesions, and sometimes especially patches dominated by small lesions or lesion borders. Almaaz et al. 2025 use lesion-aware patch sampling for MS lesions.
MDPI

The same idea is directly applicable to ATLAS: sample crops that center on small lesions more often.

Architecture tweaks tailored to small lesions
A bunch of recent stroke/lesion papers explicitly mention small lesions as a failure mode and tweak architecture or multi-stage pipelines to help:

cascade models / second-stage refinement for hard, small lesions.
PMC
+1

attention & transformer-based models to better capture small structures on ATLAS.
PMC
+1

2.3. Evaluation practices for small lesions

Instead of silently deleting small lesions, the more common, “clean” evaluation strategies are:

Report performance vs lesion size
Many stroke and MS lesion benchmarks report Dice (or other metrics) separately for small, medium, and large lesions, or at least show Dice vs lesion volume scatter plots.
ResearchGate
+1

Lesion-wise vs voxel-wise metrics
For tiny lesions, even a small misalignment kills voxel-wise Dice. Some works also report:

lesion-wise detection rates,

precision/recall for lesion centers,

or relax spatial overlap requirements.

Prediction-side connected component cleanup
A lot of pipelines remove very small isolated predicted blobs (e.g., < 3–5 voxels) as likely FPs. This is usually applied to predictions, not ground-truth, and is often used in LINDA-style and challenge-winning methods.
PMC
+1

2.4. What I’d recommend for you right now

Given that you inspected your “too small” cases and saw normal-sized lesions getting thrown out, I’d strongly avoid any heavy-handed exclusion step until it’s fully debugged.

For now:

Turn off your tiny-lesion filtering completely
→ Which is exactly what the v2 pipeline above does by ignoring _normal and operating on the full _standardized set.

Keep all lesions in training, but:

Use lesion-aware sampling (always include lesion-centric patches in batches; over-sample small lesions).

Consider a size-aware loss later (focal loss, size-reweighted Dice, or an MSL/DBL-style multi-class scheme if you feel fancy).

For evaluation, don’t delete tiny lesions:

Instead, stratify results by lesion size (e.g., quartiles by mask_ml_clean) and show Dice vs size. That’s exactly what recent ATLAS-based papers recommend to make sense of performance on small lesions.
ResearchGate
+1

Later, when you want to revisit tiny-lesion filtering, we can:

Base it on clear, very small volume thresholds (e.g., < 5–10 voxels) and

Apply it only to eval (e.g., evaluate with and without these “micro” lesions) rather than silently nuking them from your training pool.

If you want, next step we can also:

rip out the old tiny-lesion _normal-creating code, and

write a new, super-transparent “micro-lesion” flagging script that just adds a is_micro_lesion column to QC instead of moving files around. 

In [8]:
# === ATLAS MNI mask audit: recompute metrics, flag extremes, and snapshot suspicious cases ===
from pathlib import Path
import numpy as np
import pandas as pd
import nibabel as nib
import matplotlib.pyplot as plt

OUT = Path("/home/rbielski/Atlas_2/Registered/mni_1mm_ants_fixed")
assert OUT.exists(), f"Missing: {OUT}"
qc_csv_in  = OUT / "qc_summary.csv"
qc_raw = pd.read_csv(qc_csv_in)

# Paths from key
def t1_path(key):  return OUT / f"{key}_T1w_MNI.nii.gz"
def msk_path(key): return OUT / f"{key}_lesion_mask_MNI.nii.gz"

rows = []
bad_paths = []
for key in qc_raw["key"]:
    t1p, mskp = t1_path(key), msk_path(key)
    if not t1p.exists() or not mskp.exists():
        bad_paths.append(key); continue

    t1 = nib.load(str(t1p))
    m  = nib.load(str(mskp))
    tz = tuple(float(x) for x in t1.header.get_zooms()[:3])
    mz = tuple(float(x) for x in m.header.get_zooms()[:3])

    t1_shp = t1.shape[:3]; m_shp = m.shape[:3]
    t1_vox = int(np.prod(t1_shp)); m_vox = int(np.prod(m_shp))

    # “Brain size” proxy in MNI = non-zero vox in T1
    t1_data = t1.get_fdata()
    t1_nonzero = int(np.count_nonzero(t1_data > 0))

    # Make sure mask is binary
    m_data = (m.get_fdata() > 0.5).astype(np.uint8)
    m_vox_mask = int(m_data.sum())

    # Volumes
    t1_vmm3 = tz[0]*tz[1]*tz[2]
    m_vmm3  = mz[0]*mz[1]*mz[2]
    t1_fov_ml = (t1_vox * t1_vmm3) / 1000.0
    m_ml      = (m_vox_mask * m_vmm3) / 1000.0

    # Border touch?
    border_touch = bool(
        m_data[0,:,:].any() or m_data[-1,:,:].any() or
        m_data[:,0,:].any() or m_data[:,-1,:].any() or
        m_data[:,:,0].any() or m_data[:,:,-1].any()
    )

    rows.append(dict(
        key=key,
        # T1 metrics
        t1_shape_x=t1_shp[0], t1_shape_y=t1_shp[1], t1_shape_z=t1_shp[2],
        t1_zooms_x=tz[0], t1_zooms_y=tz[1], t1_zooms_z=tz[2],
        t1_vox_total=t1_vox, t1_img_nonzero=t1_nonzero,
        t1_brain_frac=(t1_nonzero/t1_vox if t1_vox else np.nan),
        t1_fov_vol_ml=t1_fov_ml,
        # Mask metrics
        m_shape_x=m_shp[0], m_shape_y=m_shp[1], m_shape_z=m_shp[2],
        m_zooms_x=mz[0], m_zooms_y=mz[1], m_zooms_z=mz[2],
        mask_voxels=m_vox_mask, mask_ml=m_ml,
        mask_border_touch=int(border_touch),
        # sanity checks
        shapes_match=int(t1_shp == m_shp),
        affines_match=int(np.allclose(t1.affine, m.affine, atol=1e-4))
    ))

df = pd.DataFrame(rows).sort_values("key").reset_index(drop=True)
out_csv = OUT / "atlas_mask_qc.csv"
df.to_csv(out_csv, index=False)
print(f"Wrote: {out_csv}")
if bad_paths:
    print("Missing T1/mask for keys:", len(bad_paths))

# Summaries
def pct(a,q): return float(np.percentile(a, q)) if len(a) else np.nan
print("\nPairs:", len(df))

# Mask volumes
m_ml = df["mask_ml"].to_numpy(float)
print("\n=== Mask volume (mL) ===")
print(f"min/median/max: {m_ml.min():.2f} / {pct(m_ml,50):.2f} / {m_ml.max():.2f}")

# Brain-size proxy from T1
nz  = df["t1_img_nonzero"].to_numpy(int)
tot = df["t1_vox_total"].to_numpy(int)
frac = df["t1_brain_frac"].to_numpy(float)
print("\n=== T1 'brain size' (nonzero voxels) ===")
print(f"min/median/max: {nz.min():,} / {int(pct(nz,50)):,} / {nz.max():,}")
print("\n=== Brain fraction of FOV ===")
print(f"mean±sd: {frac.mean():.3f} ± {frac.std():.3f} | p10/50/90: {pct(frac,10):.3f} / {pct(frac,50):.3f} / {pct(frac,90):.3f}")

# Zooms sanity (should mostly be 1,1,1)
unique_zooms = sorted(set(zip(df.m_zooms_x.round(3), df.m_zooms_y.round(3), df.m_zooms_z.round(3))))
print("\nUnique mask zooms:", unique_zooms[:5], ("… +%d more" % (len(unique_zooms)-5) if len(unique_zooms)>5 else ""))

# Flag extremes and snapshot a few
small = df[df["mask_ml"] < 1.0].copy()
large = df[df["mask_ml"] > 200.0].copy()
print(f"\nSuspicious small (<1 ml): {len(small)} | suspicious large (>200 ml): {len(large)}")

# Quick axial snapshots for the first N suspicious cases
N = 12
snap_dir = OUT / "atlas_qc_snaps"
snap_dir.mkdir(exist_ok=True)

def mid_slice(mask):
    # centroid-based slice
    idx = np.argwhere(mask > 0)
    if idx.size == 0:
        return mask.shape[2] // 2
    return int(np.median(idx[:,2]))

def draw_one(key, save_path):
    t1 = nib.load(str(t1_path(key))).get_fdata()
    m  = (nib.load(str(msk_path(key))).get_fdata() > 0.5)
    z  = mid_slice(m)
    img = t1[:,:,z]; ms = m[:,:,z]

    plt.figure(figsize=(4.6,4.6))
    plt.imshow((img - img[img>0].mean())/ (img[img>0].std() + 1e-8), cmap="gray", origin="lower")
    cs = plt.contour(ms.T, levels=[0.5], linewidths=0.8, colors="r")
    plt.title(f"{key} | slice {z}")
    plt.axis("off"); plt.tight_layout()
    plt.savefig(save_path, dpi=150); plt.close()

for key in list(small["key"].head(N)) + list(large["key"].head(N)):
    out_png = snap_dir / f"{key}.png"
    try:
        draw_one(key, out_png)
    except Exception as e:
        print("snap error:", key, e)

print(f"\nSaved snapshots to: {snap_dir}")
print("\nExamples (first 10 tiny masks):")
print(small[["key","mask_voxels","mask_ml","mask_border_touch"]].head(10).to_string(index=False))
print("\nExamples (first 10 large masks):")
print(large[["key","mask_voxels","mask_ml","mask_border_touch"]].head(10).to_string(index=False))


Wrote: /home/rbielski/Atlas_2/Registered/mni_1mm_ants_fixed/atlas_mask_qc.csv

Pairs: 655

=== Mask volume (mL) ===
min/median/max: 0.00 / 3.31 / 163.42

=== T1 'brain size' (nonzero voxels) ===
min/median/max: 3,238,496 / 4,515,338 / 8,092,277

=== Brain fraction of FOV ===
mean±sd: 0.549 ± 0.079 | p10/50/90: 0.496 / 0.529 / 0.608

Unique mask zooms: [(1.0, 1.0, 1.0)] 

Suspicious small (<1 ml): 193 | suspicious large (>200 ml): 0

Saved snapshots to: /home/rbielski/Atlas_2/Registered/mni_1mm_ants_fixed/atlas_qc_snaps

Examples (first 10 tiny masks):
               key  mask_voxels  mask_ml  mask_border_touch
sub-r001s018_ses-1          186    0.186                  0
sub-r001s022_ses-1           70    0.070                  0
sub-r001s023_ses-1           21    0.021                  0
sub-r001s024_ses-1          556    0.556                  0
sub-r001s027_ses-1          922    0.922                  0
sub-r001s028_ses-1          106    0.106                  0
sub-r001s038_ses-1    

In [11]:
import pandas as pd, pathlib
OUT = pathlib.Path("/home/rbielski/Atlas_2/Registered/mni_1mm_ants_fixed")
df  = pd.read_csv(OUT / "atlas_mask_qc.csv")
(df[df["mask_ml"] < 1.0]
   .sort_values("mask_ml")
   [["key","mask_voxels","mask_ml","mask_border_touch"]]
).to_csv(OUT / "atlas_tiny_masks.csv", index=False)
print("Wrote:", OUT / "atlas_tiny_masks.csv")


Wrote: /home/rbielski/Atlas_2/Registered/mni_1mm_ants_fixed/atlas_tiny_masks.csv


# Double check that those tiny masks arent part of a larger mask within the whole MRI 

In [12]:
# === Component audit for tiny masks (<1 ml): size distribution per case ===
from pathlib import Path
import pandas as pd, numpy as np, nibabel as nib
from scipy.ndimage import label

OUT = Path("/home/rbielski/Atlas_2/Registered/mni_1mm_ants_fixed")
qc  = pd.read_csv(OUT / "atlas_mask_qc.csv")

tiny = qc[qc["mask_ml"] < 1.0].copy()
rows = []
for _, r in tiny.iterrows():
    key = r["key"]
    mpath = OUT / f"{key}_lesion_mask_MNI.nii.gz"
    if not mpath.exists():
        continue
    m = (nib.load(str(mpath)).get_fdata() > 0.5)
    if m.sum() == 0:
        rows.append(dict(key=key, n_components=0, total_vox=0, largest_vox=0, frac_in_largest=np.nan))
        continue
    lab, n = label(m)  # 3D connected components
    sizes = np.bincount(lab.ravel())[1:]  # skip background
    total = int(sizes.sum())
    largest = int(sizes.max())
    rows.append(dict(
        key=key,
        n_components=int(n),
        total_vox=total,
        largest_vox=largest,
        frac_in_largest=float(largest/total)
    ))

df = pd.DataFrame(rows).sort_values(["largest_vox","n_components"], ascending=[False, True])
out_csv = OUT / "atlas_tiny_masks_components.csv"
df.to_csv(out_csv, index=False)
print("Wrote:", out_csv)
print(df.head(12).to_string(index=False))


Wrote: /home/rbielski/Atlas_2/Registered/mni_1mm_ants_fixed/atlas_tiny_masks_components.csv
               key  n_components  total_vox  largest_vox  frac_in_largest
sub-r010s001_ses-1             1        944          944         1.000000
sub-r015s018_ses-1             4        942          938         0.995754
sub-r005s077_ses-1             3        920          918         0.997826
sub-r003s006_ses-1             2        908          907         0.998899
sub-r001s027_ses-1             2        922          873         0.946855
sub-r011s032_ses-1             2        922          873         0.946855
sub-r009s071_ses-1             2        870          869         0.998851
sub-r015s006_ses-1             2        849          848         0.998822
sub-r031s026_ses-1             1        835          835         1.000000
sub-r034s047_ses-1             3        986          809         0.820487
sub-r046s007_ses-1             2        831          789         0.949458
sub-r009s046_ses-1  

In [ ]:
# === ATLAS T1w + lesion-masks: metrics + rich summaries + sample rows (native & MNI) ===
from pathlib import Path
import re
import numpy as np
import pandas as pd
import nibabel as nib
from math import prod
from IPython.display import display

# ---- Directories ----
ROOT    = Path("/home/rbielski/Atlas_2/Registered")
TRAIN_IMAGES = Path("/home/rbielski/Atlas_2/Training/Images")
OUT_NAT = ROOT / "native_resampled_masks"   # *_lesion_mask_T1w_native.nii.gz
OUT_MNI = ROOT / "mni_1mm_ants_fixed"       # *_T1w_MNI.nii.gz + *_lesion_mask_MNI.nii.gz
assert OUT_NAT.exists(), f"Missing {OUT_NAT}"
assert OUT_MNI.exists(), f"Missing {OUT_MNI}"

# ---- Helpers ----
def _tag(s, t):
    m = re.search(fr"({t}-[^_]+)", s)
    return m.group(1) if m else None

def key_from_name(p: Path) -> str:
    sub = _tag(p.name, "sub")
    ses = _tag(p.name, "ses")
    return "_".join([x for x in (sub, ses) if x])

def t1_pref(name: str) -> int:
    n = name.lower()
    if "tfl" in n: return 0
    if "mprage" in n: return 1
    if "mp2rage" in n: return 2
    return 3

def choose_native_t1(key: str) -> Path | None:
    cands = sorted(TRAIN_IMAGES.glob(f"{key}*T1w.nii.gz"))
    if not cands: return None
    cands.sort(key=lambda p: (t1_pref(p.name), p.name))
    return cands[0]

def img_metrics(img_path: Path, nonzero_thresh=0.0):
    img = nib.load(str(img_path))
    shp = img.shape[:3]
    z   = tuple(float(v) for v in img.header.get_zooms()[:3])
    data = img.get_fdata()
    nz  = int(np.count_nonzero(data > nonzero_thresh))
    vox_total = int(prod(shp))
    fov_mm = (shp[0]*z[0], shp[1]*z[1], shp[2]*z[2])
    voxel_vol_mm3 = z[0]*z[1]*z[2]
    fov_vol_ml = (vox_total * voxel_vol_mm3) / 1000.0
    return dict(
        shape_x=shp[0], shape_y=shp[1], shape_z=shp[2],
        zooms_x=z[0], zooms_y=z[1], zooms_z=z[2],
        vox_total=vox_total,
        img_nonzero=nz,
        brain_frac=(nz/vox_total if vox_total else np.nan),
        fov_mm_x=fov_mm[0], fov_mm_y=fov_mm[1], fov_mm_z=fov_mm[2],
        voxel_vol_mm3=voxel_vol_mm3,
        fov_vol_ml=fov_vol_ml,
    )

def mask_metrics(mask_path: Path, zooms_xyz: tuple[float,float,float]):
    m = nib.load(str(mask_path))
    vox = int(np.count_nonzero(m.get_fdata() > 0.5))
    ml  = (vox * zooms_xyz[0] * zooms_xyz[1] * zooms_xyz[2]) / 1000.0
    return dict(mask_voxels=vox, mask_ml=ml)

# ---- Build rows driven by MNI outputs ----
t1_mni_files = sorted(OUT_MNI.glob("*_T1w_MNI.nii.gz"))
assert t1_mni_files, f"No *_T1w_MNI.nii.gz found in {OUT_MNI}"

rows = []
for t1_mni in t1_mni_files:
    key = key_from_name(t1_mni)
    msk_mni = OUT_MNI / f"{key}_lesion_mask_MNI.nii.gz"
    if not msk_mni.exists():
        continue

    # Native counterparts
    t1_nat = choose_native_t1(key)
    msk_nat = OUT_NAT / f"{key}_lesion_mask_T1w_native.nii.gz"

    # MNI metrics
    mni_img = img_metrics(t1_mni, nonzero_thresh=0.0)
    mni_mask = mask_metrics(msk_mni, (mni_img["zooms_x"], mni_img["zooms_y"], mni_img["zooms_z"]))

    # Native metrics
    if t1_nat and msk_nat.exists():
        nat_img = img_metrics(t1_nat, nonzero_thresh=0.0)
        nat_mask = mask_metrics(msk_nat, (nat_img["zooms_x"], nat_img["zooms_y"], nat_img["zooms_z"]))
    else:
        nat_img = {k: np.nan for k in [
            "shape_x","shape_y","shape_z","zooms_x","zooms_y","zooms_z","vox_total",
            "img_nonzero","brain_frac","fov_mm_x","fov_mm_y","fov_mm_z","voxel_vol_mm3","fov_vol_ml"
        ]}
        nat_mask = {"mask_voxels": np.nan, "mask_ml": np.nan}
        t1_nat = None
        msk_nat = None

    rows.append({
        "key": key,
        # Native
        "nat_shape_x": nat_img["shape_x"], "nat_shape_y": nat_img["shape_y"], "nat_shape_z": nat_img["shape_z"],
        "nat_zooms_x": nat_img["zooms_x"], "nat_zooms_y": nat_img["zooms_y"], "nat_zooms_z": nat_img["zooms_z"],
        "nat_vox_total": nat_img["vox_total"], "nat_img_nonzero": nat_img["img_nonzero"], "nat_brain_frac": nat_img["brain_frac"],
        "nat_fov_mm_x": nat_img["fov_mm_x"], "nat_fov_mm_y": nat_img["fov_mm_y"], "nat_fov_mm_z": nat_img["fov_mm_z"],
        "nat_voxel_vol_mm3": nat_img["voxel_vol_mm3"], "nat_fov_vol_ml": nat_img["fov_vol_ml"],
        "nat_mask_voxels": nat_mask["mask_voxels"], "nat_mask_ml": nat_mask["mask_ml"],
        # MNI
        "mni_shape_x": mni_img["shape_x"], "mni_shape_y": mni_img["shape_y"], "mni_shape_z": mni_img["shape_z"],
        "mni_zooms_x": mni_img["zooms_x"], "mni_zooms_y": mni_img["zooms_y"], "mni_zooms_z": mni_img["zooms_z"],
        "mni_vox_total": mni_img["vox_total"], "mni_img_nonzero": mni_img["img_nonzero"], "mni_brain_frac": mni_img["brain_frac"],
        "mni_fov_mm_x": mni_img["fov_mm_x"], "mni_fov_mm_y": mni_img["fov_mm_y"], "mni_fov_mm_z": mni_img["fov_mm_z"],
        "mni_voxel_vol_mm3": mni_img["voxel_vol_mm3"], "mni_fov_vol_ml": mni_img["fov_vol_ml"],
        "mni_mask_voxels": mni_mask["mask_voxels"], "mni_mask_ml": mni_mask["mask_ml"],
    })

df = pd.DataFrame(rows).sort_values("key").reset_index(drop=True)

# Save CSV
csv_path = OUT_MNI / "image_mask_metrics.csv"
df.to_csv(csv_path, index=False)
print(f"Wrote: {csv_path}")
print("Pairs:", len(df))

# ===== Summaries =====
def summary_block(prefix: str, label: str):
    ok = df[f"{prefix}_img_nonzero"].notna()
    if not ok.any():
        print(f"\n=== {label} ===\n(no data)")
        return
    vox   = df.loc[ok, f"{prefix}_img_nonzero"].to_numpy(int)
    total = df.loc[ok, f"{prefix}_vox_total"].to_numpy(int)
    frac  = df.loc[ok, f"{prefix}_brain_frac"].to_numpy(float)
    fovml = df.loc[ok, f"{prefix}_fov_vol_ml"].to_numpy(float)
    mvox  = df.loc[ok, f"{prefix}_mask_voxels"].to_numpy(int)
    mml   = df.loc[ok, f"{prefix}_mask_ml"].to_numpy(float)

    pct = lambda a, q: np.percentile(a, q)
    print(f"\n=== {label} ===")
    print("ICV proxy (nonzero voxels):")
    print(f"  min/median/max: {vox.min():,} / {int(pct(vox,50)):,} / {vox.max():,}")
    print("Brain fraction of FOV (nonzero/total):")
    print(f"  mean±sd: {frac.mean():.3f} ± {frac.std():.3f} | p10/50/90: {pct(frac,10):.3f} / {pct(frac,50):.3f} / {pct(frac,90):.3f}")
    print("FOV volume (mL):")
    print(f"  min/median/max: {fovml.min():.1f} / {pct(fovml,50):.1f} / {fovml.max():.1f}")
    print("Lesion mask volume (mL):")
    print(f"  min/median/max: {mml.min():.2f} / {pct(mml,50):.2f} / {mml.max():.2f}")
    print("Lesion mask voxels:")
    print(f"  min/median/max: {mvox.min():,} / {int(pct(mvox,50)):,} / {mvox.max():,}")

summary_block("nat", "NATIVE summary")
summary_block("mni", "MNI summary")

# ===== Sample tables (first 10 rows) =====
pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 220)

native_cols = [
    "key",
    "nat_shape_x","nat_shape_y","nat_shape_z",
    "nat_zooms_x","nat_zooms_y","nat_zooms_z",
    "nat_vox_total","nat_img_nonzero","nat_brain_frac",
    "nat_fov_mm_x","nat_fov_mm_y","nat_fov_mm_z","nat_fov_vol_ml",
    "nat_mask_voxels","nat_mask_ml",
]
mni_cols = [
    "key",
    "mni_shape_x","mni_shape_y","mni_shape_z",
    "mni_zooms_x","mni_zooms_y","mni_zooms_z",
    "mni_vox_total","mni_img_nonzero","mni_brain_frac",
    "mni_fov_mm_x","mni_fov_mm_y","mni_fov_mm_z","mni_fov_vol_ml",
    "mni_mask_voxels","mni_mask_ml",
]
combined_cols = [
    "key",
    "nat_shape_x","nat_shape_y","nat_shape_z","nat_zooms_x","nat_zooms_y","nat_zooms_z",
    "nat_vox_total","nat_img_nonzero","nat_mask_voxels","nat_mask_ml",
    "mni_shape_x","mni_shape_y","mni_shape_z","mni_zooms_x","mni_zooms_y","mni_zooms_z",
    "mni_vox_total","mni_img_nonzero","mni_mask_voxels","mni_mask_ml"
]

print("\n=== NATIVE (first 10 rows) ===")
display(df[native_cols].head(10))
print("\n=== MNI (first 10 rows) ===")
display(df[mni_cols].head(10))
print("\n=== COMBINED (first  10 rows) ===")
display(df[combined_cols].head(10))


Wrote: /home/rbielski/Atlas_2/Registered/mni_1mm_ants_fixed/image_mask_metrics.csv
Pairs: 655

=== NATIVE summary ===
ICV proxy (nonzero voxels):
  min/median/max: 7,929,200 / 8,255,024 / 8,675,289
Brain fraction of FOV (nonzero/total):
  mean±sd: 0.953 ± 0.012 | p10/50/90: 0.944 / 0.952 / 0.961
FOV volume (mL):
  min/median/max: 8675.3 / 8675.3 / 8675.3
Lesion mask volume (mL):
  min/median/max: 0.00 / 5.29 / 496.66
Lesion mask voxels:
  min/median/max: 0 / 5,290 / 496,656

=== MNI summary ===
ICV proxy (nonzero voxels):
  min/median/max: 3,238,496 / 4,515,338 / 8,092,277
Brain fraction of FOV (nonzero/total):
  mean±sd: 0.549 ± 0.079 | p10/50/90: 0.496 / 0.529 / 0.608
FOV volume (mL):
  min/median/max: 8530.0 / 8530.0 / 8530.0
Lesion mask volume (mL):
  min/median/max: 0.00 / 3.31 / 163.42
Lesion mask voxels:
  min/median/max: 0 / 3,314 / 163,423

=== NATIVE (first 10 rows) ===


,key,nat_shape_x,nat_shape_y,nat_shape_z,nat_zooms_x,nat_zooms_y,nat_zooms_z,nat_vox_total,nat_img_nonzero,nat_brain_frac,nat_fov_mm_x,nat_fov_mm_y,nat_fov_mm_z,nat_fov_vol_ml,nat_mask_voxels,nat_mask_ml
0,sub-r001s001_ses-1,197,233,189,1.0,1.0,1.0,8675289,8078084,0.931160,197.0,233.0,189.0,8675.289,112451,112.451
1,sub-r001s002_ses-1,197,233,189,1.0,1.0,1.0,8675289,8301302,0.956891,197.0,233.0,189.0,8675.289,132590,132.590
2,sub-r001s003_ses-1,197,233,189,1.0,1.0,1.0,8675289,8228301,0.948476,197.0,233.0,189.0,8675.289,1482,1.482
3,sub-r001s004_ses-1,197,233,189,1.0,1.0,1.0,8675289,8193356,0.944448,197.0,233.0,189.0,8675.289,44659,44.659
4,sub-r001s005_ses-1,197,233,189,1.0,1.0,1.0,8675289,8363568,0.964068,197.0,233.0,189.0,8675.289,31792,31.792
5,sub-r001s006_ses-1,197,233,189,1.0,1.0,1.0,8675289,8191210,0.944200,197.0,233.0,189.0,8675.289,3009,3.009
6,sub-r001s007_ses-1,197,233,189,1.0,1.0,1.0,8675289,8251470,0.951146,197.0,233.0,189.0,8675.289,4102,4.102
7,sub-r001s008_ses-1,197,233,189,1.0,1.0,1.0,8675289,8250914,0.951082,197.0,233.0,189.0,8675.289,2083,2.083
8,sub-r001s009_ses-1,197,233,189,1.0,1.0,1.0,8675289,8235740,0.949333,197.0,233.0,189.0,8675.289,8840,8.840
9,sub-r001s010_ses-1,197,233,189,1.0,1.0,1.0,8675289,8231306,0.948822,197.0,233.0,189.0,8675.289,108394,108.394



=== MNI (first 10 rows) ===


,key,mni_shape_x,mni_shape_y,mni_shape_z,mni_zooms_x,mni_zooms_y,mni_zooms_z,mni_vox_total,mni_img_nonzero,mni_brain_frac,mni_fov_mm_x,mni_fov_mm_y,mni_fov_mm_z,mni_fov_vol_ml,mni_mask_voxels,mni_mask_ml
0,sub-r001s001_ses-1,193,229,193,1.0,1.0,1.0,8530021,4710948,0.552279,193.0,229.0,193.0,8530.021,50218,50.218
1,sub-r001s002_ses-1,193,229,193,1.0,1.0,1.0,8530021,4406390,0.516574,193.0,229.0,193.0,8530.021,56278,56.278
2,sub-r001s003_ses-1,193,229,193,1.0,1.0,1.0,8530021,4384179,0.513970,193.0,229.0,193.0,8530.021,1078,1.078
3,sub-r001s004_ses-1,193,229,193,1.0,1.0,1.0,8530021,4580281,0.536960,193.0,229.0,193.0,8530.021,18161,18.161
4,sub-r001s005_ses-1,193,229,193,1.0,1.0,1.0,8530021,4152357,0.486793,193.0,229.0,193.0,8530.021,21949,21.949
5,sub-r001s006_ses-1,193,229,193,1.0,1.0,1.0,8530021,4531902,0.531288,193.0,229.0,193.0,8530.021,1602,1.602
6,sub-r001s007_ses-1,193,229,193,1.0,1.0,1.0,8530021,4246885,0.497875,193.0,229.0,193.0,8530.021,4252,4.252
7,sub-r001s008_ses-1,193,229,193,1.0,1.0,1.0,8530021,4331016,0.507738,193.0,229.0,193.0,8530.021,1655,1.655
8,sub-r001s009_ses-1,193,229,193,1.0,1.0,1.0,8530021,7679674,0.900311,193.0,229.0,193.0,8530.021,3727,3.727
9,sub-r001s010_ses-1,193,229,193,1.0,1.0,1.0,8530021,4332797,0.507947,193.0,229.0,193.0,8530.021,43536,43.536



=== COMBINED (first 10 rows) ===


,key,nat_shape_x,nat_shape_y,nat_shape_z,nat_zooms_x,nat_zooms_y,nat_zooms_z,nat_vox_total,nat_img_nonzero,nat_mask_voxels,nat_mask_ml,mni_shape_x,mni_shape_y,mni_shape_z,mni_zooms_x,mni_zooms_y,mni_zooms_z,mni_vox_total,mni_img_nonzero,mni_mask_voxels,mni_mask_ml
0,sub-r001s001_ses-1,197,233,189,1.0,1.0,1.0,8675289,8078084,112451,112.451,193,229,193,1.0,1.0,1.0,8530021,4710948,50218,50.218
1,sub-r001s002_ses-1,197,233,189,1.0,1.0,1.0,8675289,8301302,132590,132.590,193,229,193,1.0,1.0,1.0,8530021,4406390,56278,56.278
2,sub-r001s003_ses-1,197,233,189,1.0,1.0,1.0,8675289,8228301,1482,1.482,193,229,193,1.0,1.0,1.0,8530021,4384179,1078,1.078
3,sub-r001s004_ses-1,197,233,189,1.0,1.0,1.0,8675289,8193356,44659,44.659,193,229,193,1.0,1.0,1.0,8530021,4580281,18161,18.161
4,sub-r001s005_ses-1,197,233,189,1.0,1.0,1.0,8675289,8363568,31792,31.792,193,229,193,1.0,1.0,1.0,8530021,4152357,21949,21.949
5,sub-r001s006_ses-1,197,233,189,1.0,1.0,1.0,8675289,8191210,3009,3.009,193,229,193,1.0,1.0,1.0,8530021,4531902,1602,1.602
6,sub-r001s007_ses-1,197,233,189,1.0,1.0,1.0,8675289,8251470,4102,4.102,193,229,193,1.0,1.0,1.0,8530021,4246885,4252,4.252
7,sub-r001s008_ses-1,197,233,189,1.0,1.0,1.0,8675289,8250914,2083,2.083,193,229,193,1.0,1.0,1.0,8530021,4331016,1655,1.655
8,sub-r001s009_ses-1,197,233,189,1.0,1.0,1.0,8675289,8235740,8840,8.840,193,229,193,1.0,1.0,1.0,8530021,7679674,3727,3.727
9,sub-r001s010_ses-1,197,233,189,1.0,1.0,1.0,8675289,8231306,108394,108.394,193,229,193,1.0,1.0,1.0,8530021,4332797,43536,43.536


We’ll now standardize:

apply a common MNI brain mask per case

robust intensity normalization inside the brain

component-aware mask cleanup (optional, conservative)

write a QC CSV with before/after stats

In [13]:
# === Common preprocessing utilities for MNI-registered T1w + lesion masks ===
import sys, os, json, csv, shutil, subprocess, site
from pathlib import Path
import numpy as np
import nibabel as nib
from nibabel.processing import resample_from_to
import pandas as pd
from scipy.ndimage import label

# --- TemplateFlow brain mask (MNI152NLin2009cAsym, res-01) ---
try:
    from templateflow.api import get as tf_get
except Exception:
    subprocess.run([sys.executable, "-m", "pip", "install", "--user", "templateflow"], check=True)
    sys.path.append(site.getusersitepackages())
    from templateflow.api import get as tf_get

TPL_MASK = tf_get("MNI152NLin2009cAsym", resolution=1, suffix="mask", desc="brain", extension="nii.gz")
TPL_MASK = Path(TPL_MASK[0] if isinstance(TPL_MASK, (list, tuple)) else TPL_MASK)
assert TPL_MASK.exists(), f"Missing TemplateFlow brain mask: {TPL_MASK}"

# --- Helpers ---
def _load(path: Path) -> nib.Nifti1Image:
    return nib.load(str(path))

def _save_like(ref_img: nib.Nifti1Image, data: np.ndarray, out_path: Path, dtype=np.float32):
    img = nib.Nifti1Image(data.astype(dtype), ref_img.affine, ref_img.header)
    img.set_data_dtype(dtype)
    nib.save(img, str(out_path))

def resample_brain_mask_to(ref_img: nib.Nifti1Image) -> np.ndarray:
    """Resample the TemplateFlow brain mask to the target grid (order=0)."""
    tpl = _load(TPL_MASK)
    rs  = resample_from_to(tpl, (ref_img.shape, ref_img.affine), order=0)
    return (rs.get_fdata() > 0.5)

def robust_scale_in_brain(t1_img: nib.Nifti1Image, brain_mask: np.ndarray, clip=(1,99)) -> np.ndarray:
    """Clip to [p1,p99] inside brain, then scale to [0,1]. Outside-brain set to 0."""
    x = t1_img.get_fdata().astype(np.float32)
    x[~brain_mask] = 0.0
    brain_vals = x[brain_mask]
    if brain_vals.size == 0:
        return np.zeros_like(x, dtype=np.float32)
    p1, p99 = np.percentile(brain_vals, clip)
    x = np.clip(x, p1, p99, out=x)
    # min-max inside brain
    bmin, bmax = x[brain_mask].min(), x[brain_mask].max()
    if bmax > bmin:
        x = (x - bmin) / (bmax - bmin)
    else:
        x[:] = 0.0
    x[~brain_mask] = 0.0
    return x

def mask_binarize(mask_img: nib.Nifti1Image, thr=0.5) -> np.ndarray:
    return (mask_img.get_fdata() > thr).astype(np.uint8)

def component_stats(mask_bin: np.ndarray):
    if mask_bin.sum() == 0:
        return dict(n_components=0, total_vox=0, largest_vox=0, frac_in_largest=np.nan)
    lab, n = label(mask_bin)
    sizes = np.bincount(lab.ravel())[1:]
    total = int(sizes.sum())
    largest = int(sizes.max())
    return dict(n_components=int(n), total_vox=total, largest_vox=largest,
                frac_in_largest=float(largest/total))

def clean_components(mask_img: nib.Nifti1Image, min_vox=100) -> np.ndarray:
    m = (mask_img.get_fdata() > 0.5)
    if m.sum() == 0:
        return m.astype(np.uint8)
    lab, n = label(m)
    if n == 0:
        return m.astype(np.uint8)
    sizes = np.bincount(lab.ravel())
    keep_ids = [i for i, s in enumerate(sizes) if i != 0 and s >= min_vox]
    if not keep_ids:
        return np.zeros_like(m, dtype=np.uint8)
    keep = np.isin(lab, keep_ids)
    return keep.astype(np.uint8)

def mm3_per_voxel(img: nib.Nifti1Image) -> float:
    z = img.header.get_zooms()[:3]
    return float(z[0]*z[1]*z[2])

def qc_row(key, t1_img, t1_norm, brain_mask, msk_img, msk_clean):
    # native stats
    t1 = t1_img.get_fdata()
    nz = int(np.count_nonzero(t1))
    vox_total = int(np.prod(t1_img.shape[:3]))
    frac = nz/vox_total if vox_total else np.nan
    vmm3 = mm3_per_voxel(t1_img)
    # norm stats
    nnz = int(np.count_nonzero(t1_norm))
    # mask stats
    m_bin = (msk_img.get_fdata() > 0.5)
    m_bin_clean = msk_clean.astype(bool)
    m_vox = int(m_bin.sum())
    m_vox_clean = int(m_bin_clean.sum())
    return dict(
        key=key,
        t1_shape_x=t1_img.shape[0], t1_shape_y=t1_img.shape[1], t1_shape_z=t1_img.shape[2],
        t1_zooms=str(tuple(round(v,3) for v in t1_img.header.get_zooms()[:3])),
        vox_total=vox_total, img_nonzero=nz, brain_frac=frac,
        voxel_mm3=vmm3,
        norm_img_nonzero=nnz,
        mask_voxels=m_vox, mask_ml=(m_vox*vmm3)/1000.0,
        mask_voxels_clean=m_vox_clean, mask_ml_clean=(m_vox_clean*vmm3)/1000.0,
        brain_voxels=int(brain_mask.sum())
    )

def process_dataset(
    IN_DIR: Path,
    OUT_DIR: Path,
    key_glob="*_T1w_MNI.nii.gz",
    mask_suffix="_lesion_mask_MNI.nii.gz",
    clean_policy="component_aware",  # "none" | "all_small" | "component_aware"
    min_component_vox=100,
    tiny_ml_threshold=1.0,
    frac_major_threshold=0.8
):
    OUT_T1  = OUT_DIR / "t1_norm"
    OUT_MSK = OUT_DIR / "masks_clean"
    OUT_T1.mkdir(parents=True, exist_ok=True)
    OUT_MSK.mkdir(parents=True, exist_ok=True)

    records = []
    t1_files = sorted(IN_DIR.glob(key_glob))
    if not t1_files:
        raise RuntimeError(f"No T1s found in {IN_DIR} with pattern {key_glob}")

    def key_from(p: Path):
        s = p.name.replace("_T1w_MNI.nii.gz", "")
        # Keep ARC style "sub-XXX_ses-YYY" and ATLAS "sub-xxx_ses-1"
        return s

    for t1_path in t1_files:
        key = key_from(t1_path)
        msk_path = IN_DIR / f"{key}{mask_suffix.replace('_lesion_mask_MNI.nii.gz','')}_lesion_mask_MNI.nii.gz" \
                   if mask_suffix.endswith("_lesion_mask_MNI.nii.gz") else IN_DIR / f"{key}{mask_suffix}"
        # In both ARC and ATLAS, mask is {key}_lesion_mask_MNI.nii.gz
        msk_path = IN_DIR / f"{key}_lesion_mask_MNI.nii.gz"
        if not msk_path.exists():
            # skip if no mask
            continue

        t1_img = _load(t1_path)
        msk_img = _load(msk_path)

        # 1) Brain mask to this subject's grid
        brain_mask = resample_brain_mask_to(t1_img)

        # 2) Normalization (inside brain); set outside=0
        t1_norm = robust_scale_in_brain(t1_img, brain_mask, clip=(1,99))

        # 3) Mask cleanup (conservative, component-aware by default)
        if clean_policy == "none":
            m_clean = (msk_img.get_fdata() > 0.5).astype(np.uint8)
        elif clean_policy == "all_small":
            m_clean = clean_components(msk_img, min_vox=min_component_vox)
        else:
            # component-aware: only clean if very small & fragmented
            m_bin = (msk_img.get_fdata() > 0.5).astype(np.uint8)
            vmm3  = mm3_per_voxel(msk_img)
            total_ml = (m_bin.sum()*vmm3)/1000.0
            st = component_stats(m_bin)
            if (total_ml < tiny_ml_threshold) and (np.isfinite(st["frac_in_largest"]) and st["frac_in_largest"] < frac_major_threshold):
                m_clean = clean_components(msk_img, min_vox=min_component_vox)
            else:
                m_clean = m_bin

        # 4) Save outputs
        t1_out  = OUT_T1 / f"{key}_T1w_MNI_norm.nii.gz"
        msk_out = OUT_MSK / f"{key}_lesion_mask_MNI_clean.nii.gz"
        _save_like(t1_img, t1_norm, t1_out, dtype=np.float32)
        _save_like(msk_img, m_clean, msk_out, dtype=np.uint8)

        # 5) QC row
        rec = qc_row(key, t1_img, t1_norm, brain_mask, msk_img, m_clean)
        rec.update(component_stats((msk_img.get_fdata() > 0.5).astype(np.uint8)))
        rec.update({f"clean_{k}": v for k, v in component_stats(m_clean).items()})
        records.append(rec)

    df = pd.DataFrame(records).sort_values("key").reset_index(drop=True)
    (OUT_DIR / "preprocess_qc.csv").write_text(df.to_csv(index=False))
    print(f"Preprocess complete. Wrote: {OUT_DIR/'preprocess_qc.csv'}")
    print(f"Cases processed: {len(df)}")
    return df


In [14]:
# === ATLAS run ===
from pathlib import Path

# Inputs (your ATLAS MNI outputs)
ATLAS_MNI_IN = Path("/home/rbielski/Atlas_2/Registered/mni_1mm_ants_fixed")
# Outputs (new standardized dataset root)
ATLAS_OUT    = ATLAS_MNI_IN / "_standardized"  # creates t1_norm/ and masks_clean/

ATLAS_OUT.mkdir(parents=True, exist_ok=True)

df_atlas = process_dataset(
    IN_DIR=ATLAS_MNI_IN,
    OUT_DIR=ATLAS_OUT,
    key_glob="*_T1w_MNI.nii.gz",
    mask_suffix="_lesion_mask_MNI.nii.gz",
    clean_policy="component_aware",  # conservative
    min_component_vox=100,
    tiny_ml_threshold=1.0,
    frac_major_threshold=0.8
)

print("\nATLAS summary:")
print("Pairs:", len(df_atlas))
print("Median lesion ml (raw/clean):",
      round(df_atlas["mask_ml"].median(),2), "/",
      round(df_atlas["mask_ml_clean"].median(),2))
print("Median brain_frac (raw):", round(df_atlas["brain_frac"].median(),3))
print("Outputs:")
print("  T1 normalized →", (ATLAS_OUT / "t1_norm"))
print("  Masks cleaned →", (ATLAS_OUT / "masks_clean"))
print("  QC CSV        →", (ATLAS_OUT / "preprocess_qc.csv"))


Preprocess complete. Wrote: /home/rbielski/Atlas_2/Registered/mni_1mm_ants_fixed/_standardized/preprocess_qc.csv
Cases processed: 655

ATLAS summary:
Pairs: 655
Median lesion ml (raw/clean): 3.31 / 3.31
Median brain_frac (raw): 0.529
Outputs:
  T1 normalized → /home/rbielski/Atlas_2/Registered/mni_1mm_ants_fixed/_standardized/t1_norm
  Masks cleaned → /home/rbielski/Atlas_2/Registered/mni_1mm_ants_fixed/_standardized/masks_clean
  QC CSV        → /home/rbielski/Atlas_2/Registered/mni_1mm_ants_fixed/_standardized/preprocess_qc.csv


In [2]:
import pandas as pd
from pathlib import Path

QC = Path("/home/rbielski/Atlas_2/Registered/mni_1mm_ants_fixed/_standardized/preprocess_qc.csv")  # or ARC path
df = pd.read_csv(QC)

# Tiny (<1 mL) lesions sorted smallest→largest; tie‑break by voxel count
tiny_cases = (df.loc[df["mask_ml_clean"].notna()]
                .query("mask_ml_clean < 1.0")
                .sort_values(["mask_ml_clean","mask_voxels_clean"], ascending=[True, True])
                [["key","mask_ml_clean","mask_voxels_clean"]])

print(f"Tiny cases (<1 mL): {len(tiny_cases)}")
display(tiny_cases.head(20))

# Ordered list (smallest→largest) for exclusion
out_list = QC.parent / "exclude_tiny.txt"
tiny_cases["key"].to_csv(out_list, index=False, header=False)
print("Wrote:", out_list)
print("First 5:", list(tiny_cases["key"].head(5)))
print("Last 5:", list(tiny_cases["key"].tail(5)))

Tiny cases (<1 mL): 193


,key,mask_ml_clean,mask_voxels_clean
36,sub-r001s038_ses-1,0.000,0
134,sub-r009s015_ses-1,0.000,0
143,sub-r009s026_ses-1,0.000,0
170,sub-r009s056_ses-1,0.000,0
212,sub-r009s106_ses-1,0.000,0
226,sub-r009s122_ses-1,0.000,0
244,sub-r010s014_ses-1,0.000,0
249,sub-r010s022_ses-1,0.000,0
290,sub-r014s015_ses-1,0.000,0
448,sub-r038s007_ses-1,0.000,0


Wrote: /home/rbielski/Atlas_2/Registered/mni_1mm_ants_fixed/_standardized/exclude_tiny.txt
First 5: ['sub-r001s038_ses-1', 'sub-r009s015_ses-1', 'sub-r009s026_ses-1', 'sub-r009s056_ses-1', 'sub-r009s106_ses-1']
Last 5: ['sub-r015s018_ses-1', 'sub-r010s001_ses-1', 'sub-r040s063_ses-1', 'sub-r009s087_ses-1', 'sub-r034s047_ses-1']


# remove all of the super tiny less that 1 ml lesions into seperate folder

In [17]:
from pathlib import Path
import pandas as pd
import os, shutil

# ====== CONFIG: set this to the standardized dataset root ======
# ATLAS:
STD_ROOT = Path("/home/rbielski/Atlas_2/Registered/mni_1mm_ants_fixed/_standardized")
# ARC (uncomment to run for ARC after ATLAS):
# STD_ROOT = Path("/home/rbielski/ARC/ds004884/derivatives/aggregates/t1w_with_masks/mni_1mm_ants_fixed/_standardized")

QC_CSV   = STD_ROOT / "preprocess_qc.csv"
TINY_TXT = STD_ROOT / "exclude_tiny.txt"  # your relabeled list (one key per line)

# ====== IO helpers ======
def _link_or_copy(src: Path, dst: Path):
    dst.parent.mkdir(parents=True, exist_ok=True)
    try:
        if dst.exists():
            dst.unlink()
        os.link(src, dst)   # hardlink (saves space/time)
    except Exception:
        shutil.copy2(src, dst)  # fallback to copy

# ====== Load keys & tiny set ======
df = pd.read_csv(QC_CSV)
all_keys = sorted(df["key"].unique())

if TINY_TXT.exists():
    tiny_keys = set(k.strip() for k in TINY_TXT.read_text().splitlines() if k.strip())
else:
    # fallback: derive tiny by volume (<1 mL) from QC
    tiny_keys = set(df.loc[df["mask_ml_clean"] < 1.0, "key"].tolist())

normal_keys = [k for k in all_keys if k not in tiny_keys]

# ====== Source files ======
SRC_T1  = STD_ROOT / "t1_norm"
SRC_MSK = STD_ROOT / "masks_clean"

# Sanity: ensure we can find files
missing = []
for k in all_keys:
    if not (SRC_T1 / f"{k}_T1w_MNI_norm.nii.gz").exists():
        missing.append(f"T1 missing: {k}")
    if not (SRC_MSK / f"{k}_lesion_mask_MNI_clean.nii.gz").exists():
        missing.append(f"Mask missing: {k}")
if missing:
    print("WARN: some files missing:\n  " + "\n  ".join(missing[:10]) + ("" if len(missing)<=10 else f"\n  ... {len(missing)-10} more"))

# ====== Dest folders ======
OUT_TINY_T1   = STD_ROOT / "_tiny"   / "t1_norm"
OUT_TINY_MSK  = STD_ROOT / "_tiny"   / "masks_clean"
OUT_NORM_T1   = STD_ROOT / "_normal" / "t1_norm"
OUT_NORM_MSK  = STD_ROOT / "_normal" / "masks_clean"

# ====== Populate ======
def place(keys, out_t1, out_msk):
    placed = 0
    for k in keys:
        t1  = SRC_T1  / f"{k}_T1w_MNI_norm.nii.gz"
        msk = SRC_MSK / f"{k}_lesion_mask_MNI_clean.nii.gz"
        if t1.exists():
            _link_or_copy(t1,  out_t1  / t1.name)
        if msk.exists():
            _link_or_copy(msk, out_msk / msk.name)
        placed += 1
    return placed

n_tiny   = place(tiny_keys,  OUT_TINY_T1,  OUT_TINY_MSK)
n_normal = place(normal_keys, OUT_NORM_T1, OUT_NORM_MSK)

# ====== Report ======
print(f"Standardized root: {STD_ROOT}")
print(f"Total keys: {len(all_keys)} | tiny: {len(tiny_keys)} | normal: {len(normal_keys)}")
print(f"Placed tiny   → {OUT_TINY_T1.parent} : {n_tiny} cases")
print(f"Placed normal → {OUT_NORM_T1.parent} : {n_normal} cases")

# Show a couple examples
print("\nExamples (tiny):", list(sorted(tiny_keys))[:5])
print("Examples (normal):", list(sorted(normal_keys))[:5])


Standardized root: /home/rbielski/Atlas_2/Registered/mni_1mm_ants_fixed/_standardized
Total keys: 655 | tiny: 193 | normal: 462
Placed tiny   → /home/rbielski/Atlas_2/Registered/mni_1mm_ants_fixed/_standardized/_tiny : 193 cases
Placed normal → /home/rbielski/Atlas_2/Registered/mni_1mm_ants_fixed/_standardized/_normal : 462 cases

Examples (tiny): ['sub-r001s018_ses-1', 'sub-r001s022_ses-1', 'sub-r001s023_ses-1', 'sub-r001s024_ses-1', 'sub-r001s027_ses-1']
Examples (normal): ['sub-r001s001_ses-1', 'sub-r001s002_ses-1', 'sub-r001s003_ses-1', 'sub-r001s004_ses-1', 'sub-r001s005_ses-1']


In [18]:
from pathlib import Path
import pandas as pd, numpy as np
from math import prod

# ===== ATLAS standardized root =====
STD = Path("/home/rbielski/Atlas_2/Registered/mni_1mm_ants_fixed/_standardized")
qc  = pd.read_csv(STD / "preprocess_qc.csv")

def q(a, p): return float(np.percentile(a, p)) if len(a) else np.nan

# Shapes & voxel mm³
shapes = qc[["t1_shape_x","t1_shape_y","t1_shape_z"]].dropna().astype(int).to_numpy()
voxel = qc["voxel_mm3"].dropna().to_numpy()
fov_ml = np.array([prod(s)*voxel[i]/1000.0 for i,s in enumerate(shapes)]) if len(shapes) else np.array([])

# Brain fraction
bf = qc["brain_frac"].dropna().to_numpy()

# Lesion volumes (raw & clean) in mL
lv_raw   = qc["mask_ml"].fillna(0).to_numpy()
lv_clean = qc["mask_ml_clean"].fillna(0).to_numpy()

# Changes by cleaning
changed = (qc["mask_voxels"] != qc["mask_voxels_clean"]).fillna(False).to_numpy()

# Tiny / large thresholds (you can tweak)
tiny_thr_ml  = 1.0
large_thr_ml = 200.0
n_tiny  = int((lv_clean < tiny_thr_ml).sum())
n_large = int((lv_clean > large_thr_ml).sum())

# Shape summary strings
sx, sy, sz = shapes[:,0], shapes[:,1], shapes[:,2]
shape_min = f"{sx.min()}×{sy.min()}×{sz.min()}"
shape_med = f"{int(q(sx,50))}×{int(q(sy,50))}×{int(q(sz,50))}"
shape_max = f"{sx.max()}×{sy.max()}×{sz.max()}"
vx_uniq   = ", ".join(sorted({f"{v:.3f}" for v in voxel}))

print(f"Dataset: ATLAS")
print(f"Cases: {len(qc)}")
print(f"Voxel mm³ (unique): {vx_uniq}")
print(f"Shape (min / median / max): {shape_min} / {shape_med} / {shape_max}")

if len(fov_ml):
    print(f"FOV volume (mL): min/median/max = {fov_ml.min():.1f} / {q(fov_ml,50):.1f} / {fov_ml.max():.1f}")

if len(bf):
    print(f"Brain fraction (nonzero/total): mean±sd = {bf.mean():.3f}±{bf.std():.3f} | p10/50/90 = {q(bf,10):.3f}/{q(bf,50):.3f}/{q(bf,90):.3f}")

print(f"Lesion mL (raw):   min/median/max = {lv_raw.min():.2f} / {q(lv_raw,50):.2f} / {lv_raw.max():.2f}")
print(f"Lesion mL (clean): min/median/max = {lv_clean.min():.2f} / {q(lv_clean,50):.2f} / {lv_clean.max():.2f}")
print(f"% masks changed by cleaning: {100.0*changed.mean():.1f}%")
print(f"Tiny lesions (<{tiny_thr_ml} mL): {n_tiny} | Large lesions (>{large_thr_ml} mL): {n_large}")

# Optional: echo where data live
print("\nFolders:")
print("  T1 normalized:", STD / "t1_norm")
print("  Masks cleaned:", STD / "masks_clean")
print("  Splits:       ", STD / "_tiny", "(tiny) |", STD / "_normal", "(normal)")


Dataset: ATLAS
Cases: 655
Voxel mm³ (unique): 1.000
Shape (min / median / max): 193×229×193 / 193×229×193 / 193×229×193
FOV volume (mL): min/median/max = 8530.0 / 8530.0 / 8530.0
Brain fraction (nonzero/total): mean±sd = 0.549±0.079 | p10/50/90 = 0.496/0.529/0.608
Lesion mL (raw):   min/median/max = 0.00 / 3.31 / 163.42
Lesion mL (clean): min/median/max = 0.00 / 3.31 / 163.42
% masks changed by cleaning: 9.8%
Tiny lesions (<1.0 mL): 193 | Large lesions (>200.0 mL): 0

Folders:
  T1 normalized: /home/rbielski/Atlas_2/Registered/mni_1mm_ants_fixed/_standardized/t1_norm
  Masks cleaned: /home/rbielski/Atlas_2/Registered/mni_1mm_ants_fixed/_standardized/masks_clean
  Splits:        /home/rbielski/Atlas_2/Registered/mni_1mm_ants_fixed/_standardized/_tiny (tiny) | /home/rbielski/Atlas_2/Registered/mni_1mm_ants_fixed/_standardized/_normal (normal)
